# Merging & Joining Data
Often, data is split across multiple tables or files. Pandas lets you combine them
just like SQL — or even more flexibly!

In [1]:
import pandas as pd, numpy as np

In [2]:
# Sample DataFrames

# emp_id 5 has no match
# emp_id 4 missing in salary

emp = pd.DataFrame({"emp_id": [1, 2, 3, 4],
                    "name": ["Amit", "Riya", "Raj", "Neha"],
                    "depertment": ["Sales", "HR", "IT", "Marketing"]}
                  )
sal = pd.DataFrame({"emp_id": [1, 2, 3, 5],
                    "salary": [50000, 60000, 70000, 65000]}
                  )
print(emp)
print(sal)

   emp_id  name depertment
0       1  Amit      Sales
1       2  Riya         HR
2       3   Raj         IT
3       4  Neha  Marketing
   emp_id  salary
0       1   50000
1       2   60000
2       3   70000
3       5   65000


## Merge Like SQL: `pd.merge()`

### Inner Join (default)
only matching emp_id

In [3]:
pd.merge(emp, sal, on='emp_id')

,emp_id,name,depertment,salary
0,1,Amit,Sales,50000
1,2,Riya,HR,60000
2,3,Raj,IT,70000


### Left Join
Keeps all employees, fills NaN where no match.

In [4]:
pd.merge(emp, sal, on='emp_id', how='left')

,emp_id,name,depertment,salary
0,1,Amit,Sales,50000.0
1,2,Riya,HR,60000.0
2,3,Raj,IT,70000.0
3,4,Neha,Marketing,NaN


### Right Join
Keeps all salary, even if no employee.

In [5]:
pd.merge(emp, sal, on='emp_id', how='right')

,emp_id,name,depertment,salary
0,1,Amit,Sales,50000
1,2,Riya,HR,60000
2,3,Raj,IT,70000
3,5,NaN,NaN,65000


### Outer Join
Includes all data, fills missing with NaN .

In [6]:
all_data = pd.merge(emp, sal, on='emp_id', how='outer')
all_data

,emp_id,name,depertment,salary
0,1,Amit,Sales,50000.0
1,2,Riya,HR,60000.0
2,3,Raj,IT,70000.0
3,4,Neha,Marketing,NaN
4,5,NaN,NaN,65000.0


If the key column contains duplicates in either dataset,
the merge may create more rows than expected (row explosion).

In [7]:
all_data.shape

(5, 4)

Row counts help validate whether duplicates or missing matches exist.

## Using `indicator=True`

In [8]:
pd.merge(emp, sal, on='emp_id',
         how='outer', indicator=True)


,emp_id,name,depertment,salary,_merge
0,1,Amit,Sales,50000.0,both
1,2,Riya,HR,60000.0,both
2,3,Raj,IT,70000.0,both
3,4,Neha,Marketing,NaN,left_only
4,5,NaN,NaN,65000.0,right_only


## Merging on Different Column Names

In [9]:
# Sample DataFrames
emp = pd.DataFrame({"emp_id": [1, 2, 3, 4],
                    "name": ["Amit", "Riya", "Raj", "Neha"],
                    "depertment": ["Sales", "HR", "IT", "Marketing"]}
                  )
sal = pd.DataFrame({"id": [1, 2, 3, 5],
                    "salary": [50000, 60000, 70000, 65000]}
                  )
pd.merge(emp, sal,
         left_on='emp_id',
         right_on='id')


,emp_id,name,depertment,id,salary
0,1,Amit,Sales,1,50000
1,2,Riya,HR,2,60000
2,3,Raj,IT,3,70000


## Concatenating DataFrames
Use `pd.concat()` to stack datasets either vertically or horizontally.

### Vertical (rows)

In [10]:
# Sample DataFrames
emp1 = pd.DataFrame({"emp_id": [1, 2],
                    "name": ["Amit", "Riya"],
                    "depertment": ["Sales", "HR"]}
)
emp2 = pd.DataFrame({"emp_id": [3, 4],
                    "name": ["Raj", "Neha"],
                    "depertment": ["IT", "Marketing"]}
)
print(emp1)
print(emp1)

   emp_id  name depertment
0       1  Amit      Sales
1       2  Riya         HR
   emp_id  name depertment
0       1  Amit      Sales
1       2  Riya         HR


In [11]:
pd.concat([emp1, emp2])

,emp_id,name,depertment
0,1,Amit,Sales
1,2,Riya,HR
0,3,Raj,IT
1,4,Neha,Marketing


### Horizontal (columns)

In [12]:
# Sample DataFrames
emp = pd.DataFrame({"emp_id": [1, 2, 3, 4],
                    "name": ["Amit", "Riya", "Raj", "Neha"],
                    "depertment": ["Sales", "HR", "IT", "Marketing"]}
                  )
sal = pd.DataFrame({"id": [1, 2, 3, 4],
                    "salary": [50000, 60000, 70000, 65000]}
                  )

In [13]:
pd.concat([emp, sal], axis=1)

,emp_id,name,depertment,id,salary
0,1,Amit,Sales,1,50000
1,2,Riya,HR,2,60000
2,3,Raj,IT,3,70000
3,4,Neha,Marketing,4,65000


## `.join()` Align on index

In [14]:
# Sample DataFrames
df_left = pd.DataFrame({'A': ['A0', 'A1', 'A2']}, index=['K0', 'K1', 'K2'])
df_right = pd.DataFrame({'B': ['B0', 'B1', 'B2']}, index=['K0', 'K1', 'K3'])

df_left.join(df_right)

,A,B
K0,A0,B0
K1,A1,B1
K2,A2,NaN


In [15]:
df_right.join(df_left)

,B,A
K0,B0,A0
K1,B1,A1
K3,B2,NaN


## merge vs join
`merge()` is column-based and more flexible, while `join()` is typically index-based.

## Summary
- Use `merge()` like SQL joins ( inner , left , right , outer )
- Use `concat()` to stack DataFrames (rows or columns)
- Handle mismatched keys and indexes with care
- Merging and joining are essential for real-world projects